In [1]:
ROLE = "Destination"
import paho.mqtt.client as mqtt
import time
import numpy as np
import matplotlib.pyplot as plt
import hmac
import pymongo
import copy

import sys
sys.path.append('../../')

import src



In [ ]:
import pymongo.collection


def record(conf, ROLE ,phase):
    if src.MQTT_RX(conf=conf, role=ROLE, phase=phase, verbose=1).send_ready_and_wait_for_begin():
        file = rx.record()
        return file
    else:
        print("failed synchronization")
        return None

def process(file:str, conf:src.CONFIG, ROLE:str, collection:pymongo.collection.Collection): 
    demod = src.rx.Demodulation(conf=conf)
    pp = src.PostProcessing(file=file, conf=conf, demod=demod, role=ROLE, plot=False)

    if pp.check():
        print("Recording is correct")
        for i in range(len(pp.Frames)):
            frame = pp.frameByNumber(i)
            hard_decision,rs, SNR = demod.decode(frame)
            index = demod.detect_message_indices(received=list(hard_decision), preamble=conf.PREAMBLE, repeat=conf.PREAMBLE_REPEAT)
            if index[0] is None or index[1] is None:
                print("No preamble detected!")
                continue

            msg_hard_decision = hard_decision[index[0]:index[1]]
            print("Message: ", pp.bits_to_string(msg_hard_decision[0:-256]))
            # print("recieved MAC: ", pp.binary_list_to_hex(msg_hard_decision[-256:]))
            expected_MAC = hmac.new(conf.MAC_KEY.encode('utf-8'), msg=pp.bits_to_string(msg_hard_decision[0:-256]).encode('utf-8'), digestmod='sha256').hexdigest()
            if pp.binary_list_to_hex(msg_hard_decision[-256:]) == expected_MAC:
                print("Good MAC")
                mac = True
            else:
                print("MAC is not correct")
                mac = False

            SNR = SNR[index[0]+10:index[1]-10]
            print("SNR: ", np.nanmean(SNR))

            insert={
                'SNR': float(np.nanmean(SNR)),
                'MAC': pp.binary_list_to_hex(msg_hard_decision[-256:]),
                'message': pp.bits_to_string(msg_hard_decision[0:-256]),
                'integrity': mac,
                'time': time.time(),
                'config': copy.deepcopy(conf.config)
            }
            collection.insert_one(insert)





conf = src.CONFIG()

if conf.connectionString is None:
    print("No connection string provided")
    exit(1)

if conf.MQTT is None:
    print("No MQTT broker provided")
    exit(1)


rx = src.RX(role=ROLE,conf=conf)
myclient = pymongo.MongoClient(conf.connectionString)
mydb = myclient["MAC_1D"]

while True:
    phase = 1
    file = record(conf=conf, ROLE=ROLE, phase=1)
    if file is None:
        print("Failed to synchronize")  
        continue
    process(file=file, conf=conf, ROLE=ROLE, collection=mydb[f'{ROLE}, phase_{phase}'])

    phase = 2
    file = record(conf=conf, ROLE=ROLE, phase=2)
    process(file=file, conf=conf, ROLE=ROLE, collection=mydb[f'{ROLE}, phase_{phase}'])


[INFO] [UHD] linux; GNU C++ version 13.3.0; Boost_108300; UHD_4.7.0.0-149-g635ad362
[INFO] [B200] Detected Device: B210
[INFO] [B200] Operating over USB 3.
[INFO] [B200] Initialize CODEC control...
[INFO] [B200] Initialize Radio control...
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Setting master clock rate selection to 'automatic'.
[INFO] [B200] Asking for clock rate 16.000000 MHz... 
[INFO] [B200] Actually got clock rate 16.000000 MHz.
[INFO] [B200] Asking for clock rate 32.000000 MHz... 
[INFO] [B200] Actually got clock rate 32.000000 MHz.
[INFO] [B200] Asking for clock rate 40.000000 MHz... 
[INFO] [B200] Actually got clock rate 40.000000 MHz.


[DESTINATION RX] Publishing 'ready' for phase 1
[DESTINATION RX] Connected. Subscribing to 'mkashani/feeds/begin'
[DESTINATION RX] publishing 'error'

 Recorded Time: 4.069598913192749

number of frames: 0
# Frame check failed ...

file size is: 159989760
Size check failed ...    
[DESTINATION RX] Publishing 'ready' for phase 2
[DESTINATION RX] Connected. Subscribing to 'mkashani/feeds/begin'
